In [37]:
import sklearn
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from pathlib import Path
from Brr_functions import no_lists
import json

'''
    DOWNOADING TABLES FROM MID AND
    Making all input tables standart format. Only numBer columns. From left to right. No lists, columns for DGW added
'''

'''
    import curent year data
'''
with open(Path('mid/FPL/Team_xG.txt'), 'r') as file:
    Team_xG = pd.DataFrame(json.loads(file.read()))
    Team_xG.index = pd.to_numeric(Team_xG.index)
    TxG = no_lists(Team_xG.sort_index())[no_lists(Team_xG).columns[:3:-1]]
    
with open(Path('mid/FPL/Team_xG_Ad.txt'), 'r') as file:
    Team_xG_Ad = pd.DataFrame(json.loads(file.read()))
    Team_xG_Ad.index = pd.to_numeric(Team_xG_Ad.index)
    TxGA = no_lists(Team_xG_Ad.sort_index())[TxG.columns]
    #display(Team_xG_Ad)
    
with open(Path('mid/FPL/Team_Opponent_xG.txt'), 'r') as file:
    Team_Opponent_xG = pd.DataFrame(json.loads(file.read()))
    Team_Opponent_xG.index = pd.to_numeric(Team_Opponent_xG.index)
    TOxG = no_lists(Team_Opponent_xG.sort_index())[TxG.columns]
    
with open(Path('mid/FPL/Team_oppdefence_coeff.txt'), 'r') as file:
    Team_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    Team_oppdefence_coeff.index = pd.to_numeric(Team_oppdefence_coeff.index)
    Todc = no_lists(Team_oppdefence_coeff.sort_index())[TxG.columns]

'''
    import 2019-2020 data
'''
year = '2019-2020'
if year=='': folder = ''
else: folder = f'history/{year}/'
with open(Path(f'{folder}mid/FPL/Team_xG.txt'), 'r') as file:
    Team_xG = pd.DataFrame(json.loads(file.read()))
    Team_xG.index = pd.to_numeric(Team_xG.index)
    TxG1920 = no_lists(Team_xG.sort_index())[no_lists(Team_xG).columns[:3:-1]]
    
with open(Path(f'{folder}mid/FPL/Team_xG_Ad.txt'), 'r') as file:
    Team_xG_Ad = pd.DataFrame(json.loads(file.read()))
    Team_xG_Ad.index = pd.to_numeric(Team_xG_Ad.index)
    TxGA1920 = no_lists(Team_xG_Ad.sort_index())[TxG1920.columns]
    #display(Team_xG_Ad)
    
with open(Path(f'{folder}mid/FPL/Team_Opponent_xG.txt'), 'r') as file:
    Team_Opponent_xG = pd.DataFrame(json.loads(file.read()))
    Team_Opponent_xG.index = pd.to_numeric(Team_Opponent_xG.index)
    TOxG1920 = no_lists(Team_Opponent_xG.sort_index())[TxG1920.columns]
    
with open(Path(f'{folder}mid/FPL/Team_oppdefence_coeff.txt'), 'r') as file:
    Team_oppdefence_coeff = pd.DataFrame(json.loads(file.read()))
    Team_oppdefence_coeff.index = pd.to_numeric(Team_oppdefence_coeff.index)
    Todc1920 = no_lists(Team_oppdefence_coeff.sort_index())[TxG1920.columns]

display(TOxG1920.mean(axis=1).mean())#That's threatAllowedAv actually


165.49999999999997

In [46]:
hist_num = 6

def x_maker( df,  hist_num, df_answers, df_opp, threatAllowedAv):
    '''
        Making table X for regression where last column ia an aswer.
        Adjusted tables used for lag data. UnAdjusted for answer and opponent strength.
    '''
    X = pd.DataFrame()
    col = []

    for i in range(hist_num, 0, -1): #Creating column names for lag columns
        col.append(f'GW-{i}')

    for i in range(0,len(df.columns)-hist_num):
        #print(i+1)
        next_tour2add = df[df.columns[i:i+6]].copy()#no_lists(df.sort_index())[df.columns[i:i+6]]
        next_tour2add.columns=col#['GW-6','GW-5','GW-4','GW-3','GW-2','GW-1']
        next_tour2add['Opponent'] = (df_opp[df_opp.columns[hist_num+i]]*threatAllowedAv)#.copy()
        next_tour2add['GW'] = df_answers[df_answers.columns[hist_num+i]]#.copy()#.tolist()
        X = X.append(next_tour2add, ignore_index=True, sort=False)
    return X.dropna()

hist_num = 6

df = TxGA1920
df_answers = TxG1920
df_opp = Todc1920
threatAllowedAv = TOxG1920.mean(axis=1).mean()

X3 = x_maker(df,  hist_num, df_answers, df_opp, threatAllowedAv)
X = X3[X3.columns[:-1]]
Y = X3['GW']
reg = LinearRegression().fit(X, Y)
display(reg.score(X, Y))
display(reg.coef_)

X0 = x_maker(TxGA,  hist_num, TxG, Todc, TOxG.mean(axis=1).mean())
X = X0[X0.columns[:-1]]
Y = X0['GW']
reg = LinearRegression().fit(X, Y)
display(reg.score(X, Y))
display(reg.coef_)

0.3464864555287659

array([0.13188576, 0.14941538, 0.08890991, 0.07163688, 0.18465726,
       0.03613278, 1.1489208 ])

0.45016043935941447

array([-0.09337645,  0.30169138,  0.18484037,  0.11834922, -0.08609683,
        0.19804672,  1.18328029])

In [29]:
X3 = X3.dropna()
X = X3[X3.columns[:-1]]
Y = X3['GW']
reg = LinearRegression().fit(X, Y)
display(reg.score(X, Y))
display(reg.coef_)

0.3464864555287659

array([0.13188576, 0.14941538, 0.08890991, 0.07163688, 0.18465726,
       0.03613278, 1.1489208 ])